In [1]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from src.data_loader import load_raw_data

from src.preprocessing import (
    drop_high_missing_columns,
    drop_identifier_columns,
    encode_m_features,
    encode_categorical_features,
    impute_missing_values,
    transform_features,
    scale_features,
    save_preprocessed_data,
    save_encoders,
)

## Load Raw Data

In [4]:
df = load_raw_data()

In [5]:
df.shape

(590540, 434)

In [6]:
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


## Missing Value Removal

In [7]:
df, dropped_cols = drop_high_missing_columns(df)


Dropping columns with > 90.0% missing values
Columns dropped  : 12
Columns kept     : 422
Dropped list     : ['dist2', 'D7', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']


In [8]:
df.shape

(590540, 422)

## Remove Identifier

In [9]:
df = drop_identifier_columns(df)


Dropping identifier columns
Dropped : ['TransactionID']


In [10]:
df.shape

(590540, 421)

## Encode M Features

In [11]:
df = encode_m_features(df)


Encoding M features
Binary encoded   : ['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9']
M4 kept for Label Encoding


In [12]:
df[["M1", "M2", "M3", "M5", "M6", "M7", "M8", "M9"]].head()

,M1,M2,M3,M5,M6,M7,M8,M9
0,1.0,1.0,1.0,0.0,1.0,NaN,NaN,NaN
1,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN
2,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
3,NaN,NaN,NaN,1.0,0.0,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Encoding Categorical Features

In [13]:
df, encoders = encode_categorical_features(df, fit=True)


Encoding categorical features
Categorical columns found : 21
Columns : ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M4', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [14]:
df[['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M4', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']].head()

,ProductCD,card4,card6,P_emaildomain,R_emaildomain,M4,id_12,id_15,id_16,id_28,...,id_30,id_31,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,4,1,1,31,31,2,2,3,2,2,...,74,96,260,4,2,2,2,2,1,1735
1,4,2,1,16,31,0,2,3,2,2,...,74,96,260,4,2,2,2,2,1,1735
2,4,4,2,36,31,0,2,3,2,2,...,74,96,260,4,2,2,2,2,1,1735
3,4,2,2,54,31,0,2,3,2,2,...,74,96,260,4,2,2,2,2,1,1735
4,1,2,1,16,31,3,1,1,1,1,...,7,124,164,3,1,0,1,1,2,954


In [15]:
df.dtypes.value_counts()

float64    397
int64       24
Name: count, dtype: int64

## Impute Missing Values

In [16]:
df.isnull().sum().sum()

np.int64(100802742)

In [17]:
df, impute_values = impute_missing_values(df, fit=True)


Imputing missing values
Missing values remaining : 0


In [18]:
df.isnull().sum().sum()

np.int64(0)

## Transform Features

In [19]:
df = transform_features(df)


Applying feature transformations
TransactionAmt → log1p applied


In [20]:
df["TransactionAmt"].describe()

count    590540.000000
mean          4.382960
std           0.937183
min           0.223943
25%           3.791459
50%           4.245190
75%           4.836282
max          10.371564
Name: TransactionAmt, dtype: float64

## Scaling Dataset

In [21]:
# Create scaled dataset
df_scaled, scaler = scale_features(df.copy(), fit=True)


Applying Standard Scaling
Columns to scale : 420
Scaling complete


## Saving Datasets and Artifacts

In [22]:
# Save original dataset

save_preprocessed_data(df, "train_preprocessed.parquet")


Preprocessed data saved
Path  : D:\Projects\credit-card-fraud-detection\data\preprocessed\train_preprocessed.parquet
Shape : (590540, 421)


In [23]:
# Save scaled dataset

save_preprocessed_data(df_scaled, "train_preprocessed_scaled.parquet")


Preprocessed data saved
Path  : D:\Projects\credit-card-fraud-detection\data\preprocessed\train_preprocessed_scaled.parquet
Shape : (590540, 421)


In [24]:
# Save preprocessing artifacts

save_encoders(
    encoders=encoders,
    impute_values=impute_values,
    dropped_cols=dropped_cols,
    scaler=scaler
)

Scaler saved        : models/scaler.pkl
Encoders saved      : models/label_encoders.pkl
Impute vals saved   : models/impute_values.pkl
Dropped cols saved  : models/dropped_columns.pkl
